# Multi-cue ablation

Runs the video multi-cue pipeline (Farneback motion + MediaPipe face + MediaPipe hands)
with ChildID-grouped CV for each ablation variant:

- `(i)` cues alone → `cue_motion`, `cue_face`, `cue_hands`
- `(ii)` concat
- `(iii)` attention only
- `(iv)` gating only
- `(v)` full (attention + gating)

**Entrypoint:** `scripts/run_multimodal_ablation.py`  
**Outputs:** `outputs/ablation_multicue/`

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

plt.switch_backend("Agg")

ROOT = Path("..").resolve()
OUT = ROOT / "outputs" / "ablation_multicue"
FIG = Path("figures")
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

VARIANT_ORDER = [
    "cue_motion", "cue_face", "cue_hands",
    "concat", "attention_only", "gating_only", "full",
]
print("ROOT:", ROOT)
print("Ablation output:", OUT)

## 1. Run ablation CV

Set `RUN_VIDEO_ABLATION = True` to extract cues from `data/videos/` and train all variants.
Use `SMOKE_MAX_VIDEOS > 0` for a short smoke test.

In [ ]:
RUN_VIDEO_ABLATION = True
SMOKE_MAX_VIDEOS = 0  # e.g. 80 for a quick check; 0 = full cohort

if RUN_VIDEO_ABLATION:
    cmd = [
        sys.executable,
        str(ROOT / "scripts" / "run_multimodal_ablation.py"),
        "--results-dir", str(OUT),
        "--feature-cache", str(ROOT / "data" / "cache" / "multimodal_features.npz"),
    ]
    if SMOKE_MAX_VIDEOS:
        cmd += ["--max-videos", str(SMOKE_MAX_VIDEOS)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
else:
    print("Skipped training. Expect an existing", OUT / "ablation_summary.csv")

## 2. Load and plot ablation summary

In [ ]:
summary_path = OUT / "ablation_summary.csv"
assert summary_path.exists(), (
    f"Missing {summary_path}. Set RUN_VIDEO_ABLATION=True or run:\n"
    "  python scripts/run_multimodal_ablation.py"
)
summary = pd.read_csv(summary_path)

# Prefer a stable display order when those columns exist
if "variant" in summary.columns:
    present = [v for v in VARIANT_ORDER if v in set(summary["variant"])]
    if present:
        summary = summary.set_index("variant").loc[present].reset_index()

summary

In [ ]:
acc_col = "mean_accuracy" if "mean_accuracy" in summary.columns else None
sd_col = "sd_accuracy" if "sd_accuracy" in summary.columns else None
label_col = "variant" if "variant" in summary.columns else summary.columns[0]

if acc_col is None:
    raise ValueError(f"No mean_accuracy column in {summary_path}. Columns={list(summary.columns)}")

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(summary))
y = summary[acc_col].to_numpy()
yerr = summary[sd_col].fillna(0).to_numpy() if sd_col else None
colors = ["#0072B2" if str(v) == "full" else "#56B4E9" for v in summary[label_col]]
ax.bar(x, y, yerr=yerr, color=colors, alpha=0.9, capsize=4, width=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(summary[label_col], rotation=25, ha="right")
ax.set_ylabel("Accuracy")
ax.set_title("Multi-cue ablation")
fig.tight_layout()
fig_path = FIG / "multicue_ablation_video.png"
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print("Wrote", fig_path)

## 3. CLI equivalents

```bash
# Full ablation
python scripts/run_multimodal_ablation.py

# Single variant
python scripts/run_multimodal.py --ablation full
python scripts/run_multimodal.py --ablation concat
python scripts/run_multimodal.py --ablation gating_only
python scripts/run_multimodal.py --ablation attention_only
python scripts/run_multimodal.py --ablation cue_motion
python scripts/run_multimodal.py --ablation cue_face
python scripts/run_multimodal.py --ablation cue_hands
```